In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (..., n_q, d_k)   K: (..., n_k, d_k)   V: (..., n_k, d_v)
    mask: broadcastable to (..., n_q, n_k); 1 = allowed, 0 = blocked
    returns output (..., n_q, d_v) and weights (..., n_q, n_k)
    """
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)       # (..., n_q, n_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    weights = F.softmax(scores, dim=-1)                       # rows sum to 1
    return weights @ V, weights


if __name__ == "__main__":
    Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
    K = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
    V = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])
    out, w = scaled_dot_product_attention(Q, K, V)
    print("weights\n", w.round(decimals=3))
    print("output\n", out.round(decimals=2))

    causal = torch.tril(torch.ones(3, 3))
    out_c, w_c = scaled_dot_product_attention(Q, K, V, causal)
    print("causal weights\n", w_c.round(decimals=3))
    print("causal output\n", out_c.round(decimals=2))

weights
 tensor([[0.4010, 0.1980, 0.4010],
        [0.1980, 0.4010, 0.4010],
        [0.2480, 0.2480, 0.5030]])
output
 tensor([[1.2000, 0.8000],
        [0.8000, 1.2000],
        [1.0000, 1.0000]])
causal weights
 tensor([[1.0000, 0.0000, 0.0000],
        [0.3300, 0.6700, 0.0000],
        [0.2480, 0.2480, 0.5030]])
causal output
 tensor([[2.0000, 0.0000],
        [0.6600, 1.3400],
        [1.0000, 1.0000]])
